# Embeddings and FAISS Vector Store for Medical Q&A

## Objectives
- Load and validate the cleaned labeled dataset.
- Build exactly 10,000 formatted Q&A text chunks.
- Generate dense sentence embeddings with `sentence-transformers/all-MiniLM-L6-v2`.
- Build a `faiss.IndexFlatL2` index and add embeddings.
- Persist both FAISS index and text mapping to disk.
- Run a sanity-check retrieval with 5 medical test queries (top-3 results each).

In [9]:
import os
import pickle
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## 1) Data Loading and Q&A Chunk Preparation

We combine each question and answer into a single retrieval unit:

`Question: ...\nAnswer: ...`

This preserves concise context for semantic search and ensures each vector maps to one readable chunk.

In [10]:
# Resolve dataset path robustly to handle possible filename case variations.
data_path = "..\data\processed\pubmedqa_cleaned_Labeled.csv"


df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Total rows available: {len(df):,}")

required_columns = {"question", "answer"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep only rows with non-empty question and answer text.
df = df.dropna(subset=["question", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["answer"] != "")]

if len(df) < 10_000:
    raise ValueError(
        f"Dataset contains only {len(df):,} valid Q&A pairs; at least 10,000 are required."
    )

# Isolate exactly 10,000 Q&A pairs.
df_subset = df.iloc[:10_000].copy().reset_index(drop=True)

# Build retrieval chunks used for embedding + mapping.
df_subset["text_chunk"] = (
    "Question: " + df_subset["question"] + "\nAnswer: " + df_subset["answer"]
)

text_chunks = df_subset["text_chunk"].tolist()
print(f"Prepared text chunks: {len(text_chunks):,}")
print("Sample chunk:\n")
print(text_chunks[0][:500])

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ZiadE\AppData\Local\Temp\ipykernel_27364\3107194924.py:2: SyntaxWarning: invalid escape sequence '\d'
  data_path = "..\data\processed\pubmedqa_cleaned_Labeled.csv"


Loaded dataset from: ..\data\processed\pubmedqa_cleaned_Labeled.csv
Total rows available: 10,000
Prepared text chunks: 10,000
Sample chunk:

Question: Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?
Answer: Naturopathy appears to be an effective alternative for relief of specific menopausal symptoms compared to conventional therapy.


## 2) Embedding Generation

We use a compact transformer model (`all-MiniLM-L6-v2`) to convert each text chunk into a dense vector.

Important implementation detail:
- FAISS expects vectors in a contiguous `float32` NumPy array.
- Converting explicitly avoids dtype mismatches and runtime errors when adding vectors to the index.

In [11]:
# Load embedding model once and reuse it for documents + queries.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
print(f"Loaded model: {model_name}")

# encode() returns embeddings; convert_to_numpy=True gives a NumPy matrix directly.
embeddings = model.encode(
    text_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# FAISS requires float32 vectors for efficient computation and compatibility.
embeddings = np.asarray(embeddings, dtype=np.float32)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embeddings dtype: {embeddings.dtype}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\ML\Python\ML_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ZiadE\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (10000, 384)
Embeddings dtype: float32


## 3) FAISS Index Construction

`IndexFlatL2` is a simple exact nearest-neighbor index using L2 distance.

Why this setup:
- Great baseline for correctness and debugging.
- No training step needed.
- Dynamic dimension (`d`) is inferred from the embedding matrix, so the code remains model-agnostic.

In [12]:
# Dynamically infer embedding dimension from generated vectors.
d = embeddings.shape[1]

# IndexFlatL2 stores vectors and performs exact search using squared L2 distance.
index = faiss.IndexFlatL2(d)

# Add all 10,000 vectors to the FAISS index.
index.add(embeddings)

print(f"FAISS index dimension: {d}")
print(f"Total vectors in index: {index.ntotal:,}")

FAISS index dimension: 384
Total vectors in index: 10,000


## 4) Persist Index and Text Mapping to Disk

FAISS stores vectors only, not source text.

To make retrieval human-readable, we also persist a chunk mapping file where each row/index maps back to:
- original row id,
- question,
- answer,
- combined `text_chunk`.

In [13]:
# Create output directory for FAISS artifacts.
output_dir = Path("data/embeddings/faiss_index")
output_dir.mkdir(parents=True, exist_ok=True)

index_path = output_dir / "pubmedqa_index_flatl2.faiss"
mapping_csv_path = output_dir / "chunk_mapping.csv"
mapping_pkl_path = output_dir / "chunk_mapping.pkl"

# Persist FAISS index binary.
faiss.write_index(index, str(index_path))

# Persist mapping table so retrieved ids can be converted back to text.
mapping_df = df_subset[["question", "answer", "text_chunk"]].copy()
mapping_df.insert(0, "chunk_id", np.arange(len(mapping_df), dtype=np.int32))
mapping_df.to_csv(mapping_csv_path, index=False)

# Optional pickle export for fast Python-native loading.
with open(mapping_pkl_path, "wb") as f:
    pickle.dump(mapping_df, f)

print(f"Saved FAISS index to: {index_path}")
print(f"Saved chunk mapping CSV to: {mapping_csv_path}")
print(f"Saved chunk mapping PKL to: {mapping_pkl_path}")

Saved FAISS index to: data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
Saved chunk mapping CSV to: data\embeddings\faiss_index\chunk_mapping.csv
Saved chunk mapping PKL to: data\embeddings\faiss_index\chunk_mapping.pkl


## 5) Retrieval Sanity Check (Top-3)

We test with five medical-domain queries, embed them with the same model, and search the FAISS index.

Interpretation note:
- `IndexFlatL2` returns squared L2 distances.
- Lower distance means higher similarity.

In [14]:
test_queries = [
    "What are effective treatments for irritable bowel syndrome symptoms?",
    "Can hypothyroidism increase risk of fatty liver disease?",
    "Is laparoscopic prostatectomy superior to retropubic surgery?",
    "How accurate is diagnosis of acute otitis media in primary care?",
    "Does secondary isoniazid therapy reduce recurrent tuberculosis in HIV patients?",
]

# Encode queries with the same model and cast to float32 for FAISS compatibility.
query_embeddings = model.encode(
    test_queries,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

k = 3  # top-3 nearest neighbors
D, I = index.search(query_embeddings, k)

for qi, query in enumerate(test_queries):
    print("=" * 110)
    print(f"Test Query {qi + 1}: {query}\n")

    for rank in range(k):
        retrieved_idx = int(I[qi, rank])
        distance = float(D[qi, rank])
        chunk_text = mapping_df.loc[retrieved_idx, "text_chunk"]

        print(f"Top {rank + 1} | Chunk ID: {retrieved_idx} | L2 Distance: {distance:.6f}")
        print(chunk_text[:700])
        if len(chunk_text) > 700:
            print("... [truncated]")
        print("-" * 110)

print("Sanity check retrieval completed.")

Test Query 1: What are effective treatments for irritable bowel syndrome symptoms?

Top 1 | Chunk ID: 6053 | L2 Distance: 0.941691
Question: Is childhood abuse or neglect associated with symptom reports and physiological measures in women with irritable bowel syndrome?
Answer: Women with IBS who self-report childhood abuse/neglect are more likely to report disturbed sleep, somatic symptoms, and psychological distress. Women with IBS should be screened for adverse childhood events including abuse/neglect.
--------------------------------------------------------------------------------------------------------------
Top 2 | Chunk ID: 1826 | L2 Distance: 0.955966
Question: Does routine abdominal ultrasound enhance diagnostic accuracy in irritable bowel syndrome?
Answer: This study confirms that a positive approach to diagnosing IBS is a safe policy. Furthermore, routine ultrasound scanning in IBS is unnecessary and could be counter-productive by detecting many minor abnormalities, which ca